# Godot DQN Agent

This notebook will host the Python-side WebSocket server and the Deep Q-Network agent.

In [ ]:
import asyncio
import websockets
import json
import torch
import numpy as np

class GodotEnv:
    def __init__(self, host='127.0.0.1', port=11000):
        self.host = host
        self.port = port
        self.server = None
        self.websocket = None
        
    async def start_server(self):
        self.server = await websockets.serve(self.handler, self.host, self.port)
        print(f"WebSocket server started at ws://{self.host}:{self.port}")
        
    async def handler(self, websocket, path):
        print("Godot connected!")
        self.websocket = websocket
        try:
            async for message in websocket:
                # Handle incoming state from Godot
                pass
        except websockets.exceptions.ConnectionClosed:
            print("Godot disconnected.")
            self.websocket = None
            
    async def step(self, action: int):
        if self.websocket:
            await self.websocket.send(json.dumps({"action": action}))
            # Wait for response (state, reward, done)
            response = await self.websocket.recv()
            data = json.loads(response)
            return data.get('state', []), data.get('reward', 0.0), data.get('done', False), data.get('score', 0.0)
        return [], 0.0, False, 0.0

    async def reset(self):
        if self.websocket:
            await self.websocket.send(json.dumps({"command": "reset"}))
            response = await self.websocket.recv()
            data = json.loads(response)
            return data.get('state', [])
        return []
